In [ ]:
sourceData='/bettik/PROJECTS/pr-regional-climate/santolam/'
model='aphro'
fileName='APHRO_mon_MA_025deg_V1101_EXR1.1951-2015.nc'
var='pre'

import numpy as np
import xarray as xr
import sys

ds0= xr.open_dataset(sourceData+model+'/'+fileName)

#It is only needed to change name of variable and eventually,the name of coordinates:
#var_obs.latitude to var_obs.lat
#NOTE: write x,y in minus if not it cannot read cdo

#Creating the 2d grid

var_obs=ds0['precip']
lat2d, lon2d = np.meshgrid(var_obs.latitude, var_obs.longitude)

lat2D = xr.DataArray(data=lat2d.transpose(),  dims=["y", "x"],
    coords=dict(x=(["x"], var_obs.longitude.values), y=(["y"], var_obs.latitude.values)), name='lat')

lon2D = xr.DataArray(data=lon2d.transpose(),  dims=["y", "x"],
    coords=dict(x=(["x"], var_obs.longitude.values), y=(["y"], var_obs.latitude.values)), name='lon')
lonlat = xr.merge([lat2D, lon2D])
lonlat.to_netcdf(sourceData+model+'/'+'grid_'+var+'_'+model+'_2d_undef.nc')

##Opening the undefined grid to add dummy variables and attributes
ds=xr.open_dataset(sourceData+model+'/'+'grid_'+var+'_'+model+'_2d_undef.nc')

# Create dummy variable based on lat (or any constant field; as in make_gdf.sh C.Amory)
dummy = xr.DataArray(
    np.ones_like(ds['lat']),
    dims=("y", "x"),
    coords={"y": ds["y"], "x": ds["x"]},
    name="dummy"
)

# Assign required attributes
dummy.attrs = {
    "long_name": "dummy variable",
    "standard_name": "latitude",
    "units": "1",
    "valid_range": (-1.e+20, 1.e+20),
    "coordinates": "lon lat"
}

# Lat and lon attributes
ds["lat"].attrs = {
    "units": "degrees_north",
    "long_name": "grid center latitude",
    "standard_name": "latitude",
    "valid_range": (-1.e+20, 1.e+20),
    "actual_range": (float(ds["lat"].min()), float(ds["lat"].max()))
}

ds["lon"].attrs = {
    "units": "degrees_east",
    "long_name": "grid center longitude",
    "standard_name": "longitude",
    "valid_range": (-1.e+20, 1.e+20),
    "actual_range": (float(ds["lon"].min()), float(ds["lon"].max()))
}

# opt: set x/y metadata
ds["x"].attrs = {
    "units": "km",
    "long_name": "X",
    "standard_name": "X"
}
ds["y"].attrs = {
    "units": "km",
    "long_name": "Y",
    "standard_name": "Y"
}

# Add the dummy variable to the dataset
ds["dummy"] = dummy

# Set global attributes
ds.attrs = {
    "title": "Grid file generated for CDO remapping - MAR model",
    "institution": "Generated via Python meshgrid",
    "history": "Created for remapping using cdo remapbil: "
}

# Save to new file
ds.to_netcdf(sourceData+model+'/'+'grid_'+var+'_'+model+'_2d_def.nc')
print(sourceData+model+'/'+'grid_'+var+'_'+model+'_2d_def.nc')
